In [1]:
import hail as hl
import os
from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import variants
from ko_utils import qc
from ko_utils import analysis

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_debug_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe051.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_debug_export.log


In [6]:
input_phased_path = "data/mt/ukb_wes_200k_annotated_chr16.mt"
input_unphased_path = "data/mt/ukb_wes_200k_annotated_chr16_singletons.mt"

In [7]:
mt1 = qc.get_table(input_path=input_phased_path, input_type='mt') 
mt2 = qc.get_table(input_path=input_unphased_path, input_type='mt')

In [8]:
mt1 = mt1.explode_rows(mt1.consequence.vep.worst_csq_by_gene_canonical)
mt2 = mt2.explode_rows(mt2.consequence.vep.worst_csq_by_gene_canonical)    

In [9]:
by_gene_annotation1 = analysis.annotation_case_builder(mt1.consequence.vep.worst_csq_by_gene_canonical, mt1.consequence.dbnsfp, use_loftee = False)
by_gene_annotation2 = analysis.annotation_case_builder(mt2.consequence.vep.worst_csq_by_gene_canonical, mt2.consequence.dbnsfp, use_loftee = False)

In [10]:
mt1_subset = mt1.annotate_rows(consequence_category = by_gene_annotation1)    
mt2_subset = mt2.annotate_rows(consequence_category = by_gene_annotation2) 

In [12]:
lst = by_gene_annotation1.collect()

In [13]:
from collections import Counter

In [14]:
Counter(lst)

Counter({'non_coding': 342230,
         'other_missense': 97576,
         'damaging_missense': 9812,
         'synonymous': 55482,
         'ptv': 7278,
         None: 11153})

In [15]:
mt1_subset = mt1_subset.filter_rows(mt1_subset.consequence_category == 'ptv')
mt2_subset = mt2_subset.filter_rows(mt2_subset.consequence_category == 'ptv')
mt1_subset.count()

(7278, 185506)

In [16]:
mt = analysis.gene_csqs_calc_pKO_pseudoSNP(mt1_subset, mt2_subset, 21)

2021-11-01 18:26:05 Hail: WARN: cols(): Resulting column table is sorted by 'col_key'.
    To preserve matrix table column order, first unkey columns with 'key_cols_by()'


In [ ]:
print(mt.count())

In [ ]:
print(mt.aggregate_entries(hl.agg.sum(~hl.is_defined(mt.DS))))

2021-11-01 18:28:22 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-01 18:30:29 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-01 18:32:33 Hail: INFO: Ordering unsorted dataset with network shuffle


In [20]:
mt.DS.show()

,,,,,,,,,,,,,,,
,,'1705888','3544034','1065590','1130642','1237482','2351928','3310264','5800421','3708144','5886032','1728817','1663370','1116994','5254480'
locus,alleles,DS,DS,DS,DS,DS,DS,DS,DS,DS,DS,DS,DS,DS,DS
locus<GRCh38>,array<str>,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64


In [15]:
result_miss = (mt.group_cols_by(mt.s).aggregate(missing=hl.agg.sum(hl.is_missing(mt.DS))))

In [16]:
result_miss.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'missing': int64
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------
